## Step 1: Import Required Libraries

In [23]:
# =========================================
# SETUP: CREATE NECESSARY DIRECTORIES
# =========================================

import os

# Create plots directories if they don't exist
plot_dirs = ['../plots/preprocessing/', '../plots/clustering/', '../plots/classification/', '../data/processed/']
for dir_path in plot_dirs:
    os.makedirs(dir_path, exist_ok=True)
    
print("✓ All directories created/verified")


✓ All directories created/verified


In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


# DBSCAN Clustering for Behavioral Pattern Analysis
## Financial Risk Classification & Behavioral Pattern Clustering

In this notebook, we'll:
1. Load processed data
2. Tune DBSCAN parameters (eps & min_samples)
3. Apply DBSCAN clustering
4. Visualize clusters & outliers with PCA and t-SNE
5. Export clustered data to CSV

## Step 2: Load and Prepare Data

In [25]:
# =========================================
# STEP 2: LOAD AND PREPARE DATA
# =========================================

# Load processed data
df = pd.read_csv('../data/processed/processed_data.csv')

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Missing values before cleaning: {df.isnull().sum().sum()}")

# Drop any remaining NaN values
df = df.dropna()

print(f"Dataset shape after dropping NaN: {df.shape}")
print(f"Missing values after cleaning: {df.isnull().sum().sum()}")

print(f"\nFirst few rows:")
print(df.head())

# Separate features (X) and target (y)
X_all = df.drop('LoanApproved', axis=1)
y = df['LoanApproved']

# =========================================
# FEATURE ENGINEERING: SELECT FINANCIAL + BEHAVIORAL FEATURES
# =========================================
selected_features = ['Income', 'LoanAmount', 'CreditScore', 'Age', 'YearsExperience']

X = X_all[selected_features].copy()

print(f"\n** FEATURE ENGINEERING APPLIED **")
print(f"Original features: {X_all.shape[1]}")
print(f"  All features: {list(X_all.columns)}")
print(f"\nSelected features: {X.shape[1]}")
print(f"  Selected: {list(X.columns)}")
print(f"\nDropped non-informative features:")
features_dropped = [col for col in X_all.columns if col not in selected_features]
for col in features_dropped:
    print(f"  - {col}")

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nData types:\n{X.dtypes.value_counts()}")
print(f"Missing values in X: {X.isnull().sum().sum()}")

print(f"\nTarget distribution (LoanApproved):")
print(y.value_counts())
print(f"\nTarget percentages:")
print((y.value_counts() / len(y) * 100).round(2))


DATASET OVERVIEW
Dataset shape: (16770, 13)
Columns: ['Age', 'Income', 'LoanAmount', 'CreditScore', 'YearsExperience', 'EmploymentType', 'Education', 'MaritalStatus', 'HomeOwnership', 'DebtToIncome', 'BankruptcyHistory', 'PaymentHistory', 'LoanApproved']
Missing values before cleaning: 0
Dataset shape after dropping NaN: (16770, 13)
Missing values after cleaning: 0

First few rows:
        Age    Income  LoanAmount  CreditScore  YearsExperience  \
0  0.482105 -0.459419   -0.974577     0.921112         0.425959   
1 -0.132883 -0.467906    0.271139     1.149790        -0.206882   
2  0.657816 -0.431860   -0.542205    -0.055968         0.787583   
3  1.624227  0.575305    1.416370    -0.575691         1.510830   
4 -0.220739  1.789160   -1.357963     0.442966        -0.026071   

   EmploymentType  Education  MaritalStatus  HomeOwnership  DebtToIncome  \
0               0          4              1              2      0.491160   
1               0          0              2              0  

## Step 3: Find Optimal Epsilon with K-Distance Graph

In [26]:
# =========================================
# STEP 2.5: PREPARE DATA FOR CLUSTERING
# =========================================

print(f"Using full dataset for clustering:")
print(f"  Total samples: {len(X):,}")
print(f"  Features: {X.shape[1]}")
print(f"  All data will be used to preserve behavioral patterns")

print(f"\nLoanApproved distribution:")
approval_counts = y.value_counts()
for val, count in approval_counts.items():
    print(f"  {val}: {count:,} ({count/len(y)*100:.1f}%)")


Using full dataset for clustering:
  Total samples: 16,770
  Features: 5
  All data will be used to preserve behavioral patterns

LoanApproved distribution:
  0: 13,075 (78.0%)
  1: 3,695 (22.0%)


In [27]:
# =========================================
# STEP 3: K-DISTANCE GRAPH FOR EPS SELECTION
# =========================================


from sklearn.neighbors import NearestNeighbors


k = 5
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X)
distances, indices = neighbors_fit.kneighbors(X)

# Sort distances
distances = np.sort(distances[:, k-1], axis=0)

# Plot k-distance graph
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(distances, linewidth=2, color='steelblue')
ax.set_xlabel('Data Points sorted by distance', fontsize=12, fontweight='bold')
ax.set_ylabel(f'{k}-NN Distance', fontsize=12, fontweight='bold')
ax.set_title(f'K-Distance Graph (k={k}) - Look for the "elbow" point', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add percentile lines
p50 = np.percentile(distances, 50)
p75 = np.percentile(distances, 75)
p90 = np.percentile(distances, 90)

ax.axhline(y=p50, color='green', linestyle='--', linewidth=2, label=f'Median (50th %ile): {p50:.3f}', alpha=0.7)
ax.axhline(y=p75, color='orange', linestyle='--', linewidth=2, label=f'75th %ile: {p75:.3f}', alpha=0.7)
ax.axhline(y=p90, color='red', linestyle='--', linewidth=2, label=f'90th %ile: {p90:.3f}', alpha=0.7)

ax.legend(fontsize=11, loc='best')
plt.tight_layout()
plt.savefig('../plots/clustering/k_distance_graph.png', dpi=150, bbox_inches='tight')
plt.close()

print("="*80)
print("K-DISTANCE GRAPH ANALYSIS:")
print("="*80)
print(f"\nDistance statistics (k={k}):")
print(f"  Min:              {distances.min():.4f}")
print(f"  Median (50th %):  {p50:.4f}")
print(f"  75th percentile:  {p75:.4f}")
print(f"  90th percentile:  {p90:.4f}")
print(f"  Max:              {distances.max():.4f}")


print("="*80)

K-DISTANCE GRAPH ANALYSIS:

Distance statistics (k=5):
  Min:              0.1561
  Median (50th %):  0.3504
  75th percentile:  0.4288
  90th percentile:  0.5347
  Max:              1.3400


## Step 4: Tune DBSCAN Parameters

In [28]:
# =========================================
# STEP 4: TUNE EPS AND MIN_SAMPLES (WITH SELECTED FEATURES)
# =========================================

# Test range of eps values based on k-distance graph analysis
# Typical values for normalized data: 0.3 - 1.2
eps_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2]
min_samples_values = [5, 10, 15]

results = []

print("Testing parameter combinations...")
print("="*100)

for eps in eps_values:
    for min_samples in min_samples_values:
        # Apply DBSCAN
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X)
        
        # Count clusters and noise points
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # Calculate metrics (only if we have at least 2 clusters)
        if n_clusters > 1 and n_noise < len(labels) - 1:
            silhouette = silhouette_score(X, labels)
            davies_bouldin = davies_bouldin_score(X, labels)
        else:
            silhouette = -1
            davies_bouldin = float('inf')
        
        results.append({
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'noise_pct': (n_noise / len(labels)) * 100,
            'silhouette': silhouette,
            'davies_bouldin': davies_bouldin
        })
        
        print(f"eps={eps:3.1f}, min_samples={min_samples:2d} → Clusters: {n_clusters:2d}, Noise: {n_noise:4d} ({(n_noise/len(labels))*100:5.2f}%), "
              f"Silhouette: {silhouette:7.4f}, Davies-Bouldin: {davies_bouldin:7.4f}")

# Create results dataframe
results_df = pd.DataFrame(results)

# Filter configurations with at least 3 clusters (realistic for this dataset)
valid_configs = results_df[results_df['n_clusters'] >= 3]

print("\n" + "="*100)
print(f"Configurations with at least 3 clusters: {len(valid_configs)} / {len(results_df)}")
if len(valid_configs) > 0:
    print("\nTop 5 configurations by Silhouette Score (with 3+ clusters):")
    print(valid_configs.nlargest(5, 'silhouette')[['eps', 'min_samples', 'n_clusters', 'n_noise', 'silhouette', 'davies_bouldin']])
else:
    print("WARNING: No configurations found with 3+ clusters. Showing best overall configurations...")
    print(results_df.nlargest(5, 'silhouette')[['eps', 'min_samples', 'n_clusters', 'n_noise', 'silhouette', 'davies_bouldin']])

print("\n" + "="*100)
print("Recommendation: Choose eps and min_samples based on:")
print("  1. ✓ At least 3 clusters for meaningful segmentation")
print("  2. ✓ Some outliers (5-20% is typical for anomaly detection)")
print("  3. ✓ High silhouette score (0.3+ is decent, 0.5+ is good)")
print("  4. ✓ Low davies_bouldin index (< 1.5 is good)")
print("\n" + "="*100)


Testing parameter combinations...
eps=0.3, min_samples= 5 → Clusters: 197, Noise: 9259 (55.21%), Silhouette: -0.6062, Davies-Bouldin:  1.4800
eps=0.3, min_samples=10 → Clusters: 57, Noise: 14304 (85.30%), Silhouette: -0.5521, Davies-Bouldin:  1.9765
eps=0.3, min_samples=15 → Clusters: 14, Noise: 16402 (97.81%), Silhouette: -0.4290, Davies-Bouldin:  2.3642
eps=0.4, min_samples= 5 → Clusters: 88, Noise: 3341 (19.92%), Silhouette: -0.4376, Davies-Bouldin:  1.3958
eps=0.4, min_samples=10 → Clusters: 25, Noise: 6544 (39.02%), Silhouette: -0.4414, Davies-Bouldin:  1.6263
eps=0.4, min_samples=15 → Clusters: 17, Noise: 9061 (54.03%), Silhouette: -0.4728, Davies-Bouldin:  1.9036
eps=0.5, min_samples= 5 → Clusters: 21, Noise: 1183 ( 7.05%), Silhouette: -0.2410, Davies-Bouldin:  1.4412
eps=0.5, min_samples=10 → Clusters: 12, Noise: 2529 (15.08%), Silhouette: -0.1861, Davies-Bouldin:  1.6489
eps=0.5, min_samples=15 → Clusters:  2, Noise: 3867 (23.06%), Silhouette:  0.0583, Davies-Bouldin:  2.5214


## Step 5: Select Best Parameters and Apply DBSCAN

In [29]:
# =========================================
# STEP 5: SELECT BEST PARAMETERS & APPLY DBSCAN
# =========================================

# Strategy 1: Try configs with 3+ clusters and good silhouette
good_clusters = results_df[results_df['n_clusters'] >= 3]
good_silhouette = good_clusters[good_clusters['silhouette'] > 0.0]

if len(good_silhouette) > 0:
    best_config = good_silhouette.loc[good_silhouette['silhouette'].idxmax()]
    reason = "Best silhouette score with 3+ clusters"
# Strategy 2: Fallback to configs with some outliers (indicates meaningful structure)
elif len(good_clusters) > 0:
    with_outliers = good_clusters[good_clusters['n_noise'] > 0]
    if len(with_outliers) > 0:
        best_config = with_outliers.loc[with_outliers['silhouette'].idxmax()]
        reason = "Best config with outliers and 3+ clusters"
    else:
        best_config = good_clusters.loc[good_clusters['silhouette'].idxmax()]
        reason = "Best silhouette with 3+ clusters"
# Strategy 3: Last resort - any valid result
else:
    best_config = results_df.loc[results_df['silhouette'].idxmax()]
    reason = "Best overall silhouette score"

best_eps = best_config['eps']
best_min_samples = int(best_config['min_samples'])

print("="*80)
print("SELECTED PARAMETERS:")
print(f"  eps = {best_eps}")
print(f"  min_samples = {best_min_samples}")
print(f"  Reason: {reason}")
print("="*80)

# Apply DBSCAN with best parameters
dbscan_final = DBSCAN(eps=best_eps, min_samples=best_min_samples)
clusters = dbscan_final.fit_predict(X)

# Extract statistics
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)

print(f"\nClustering Results:")
print(f"  Number of clusters: {n_clusters}")
print(f"  Number of noise points (outliers): {n_noise} ({(n_noise/len(clusters))*100:.2f}%)")
print(f"  Number of core points: {len(clusters) - n_noise}")

# Cluster distribution
print(f"\nCluster distribution:")
unique, counts = np.unique(clusters, return_counts=True)
for cluster_id, count in zip(unique, counts):
    if cluster_id == -1:
        print(f"  Outliers: {count:,} points ({(count/len(clusters))*100:.2f}%)")
    else:
        print(f"  Cluster {int(cluster_id):2d}: {count:,} points ({(count/len(clusters))*100:.2f}%)")

# Calculate metrics
silhouette = silhouette_score(X, clusters) if n_clusters > 1 else -1
davies_bouldin = davies_bouldin_score(X, clusters) if n_clusters > 1 else float('inf')
print(f"\nQuality Metrics:")
print(f"  Silhouette Score: {silhouette:.4f}")
print(f"  Davies-Bouldin Index: {davies_bouldin:.4f}")


SELECTED PARAMETERS:
  eps = 0.6
  min_samples = 5
  Reason: Best config with outliers and 3+ clusters

Clustering Results:
  Number of clusters: 10
  Number of noise points (outliers): 429 (2.56%)
  Number of core points: 16341

Cluster distribution:
  Outliers: 429 points (2.56%)
  Cluster  0: 16,297 points (97.18%)
  Cluster  1: 8 points (0.05%)
  Cluster  2: 4 points (0.02%)
  Cluster  3: 5 points (0.03%)
  Cluster  4: 6 points (0.04%)
  Cluster  5: 5 points (0.03%)
  Cluster  6: 5 points (0.03%)
  Cluster  7: 5 points (0.03%)
  Cluster  8: 3 points (0.02%)
  Cluster  9: 3 points (0.02%)

Quality Metrics:
  Silhouette Score: -0.0349
  Davies-Bouldin Index: 1.5981


## Step 6: Visualize Clusters with PCA

In [30]:
# =========================================
# STEP 6: VISUALIZE CLUSTERS WITH PCA
# =========================================

# Apply PCA to reduce to 2D for visualization
print("Applying PCA for 2D visualization...")
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")

# Create scatter plot
fig, ax = plt.subplots(figsize=(14, 10))

# Plot clusters
unique_clusters = set(clusters)
colors = plt.cm.tab20(np.linspace(0, 1, max(len(unique_clusters), 10)))

for idx, cluster_id in enumerate(sorted(unique_clusters)):
    if cluster_id == -1:
        # Plot outliers with X marker
        mask = clusters == cluster_id
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c='red', marker='X', s=300, alpha=0.8, 
                   edgecolors='darkred', linewidth=2,
                   label=f'Outliers ({mask.sum()} points)', zorder=5)
    else:
        # Plot clusters
        mask = clusters == cluster_id
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=[colors[idx]], s=100, alpha=0.7, 
                   edgecolors='black', linewidth=0.5,
                   label=f'Cluster {int(cluster_id)} ({mask.sum()} points)', zorder=3)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12, fontweight='bold')
ax.set_title(f'DBSCAN Clustering - PCA Visualization\neps={best_eps}, min_samples={best_min_samples}', 
             fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../plots/clustering/dbscan_clusters_pca.png', dpi=150, bbox_inches='tight')
plt.close()

print("✓ PCA visualization saved!")

Applying PCA for 2D visualization...
PCA explained variance: 63.95%
  PC1: 43.87%
  PC2: 20.08%
✓ PCA visualization saved!


## Step 7: Visualize Clusters with t-SNE

In [31]:
# =========================================
# STEP 7: VISUALIZE CLUSTERS WITH t-SNE
# =========================================

# Apply t-SNE to reduce to 2D (better for cluster separation)
print("Applying t-SNE for 2D visualization (this may take 30-60 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X)//3), max_iter=1000, verbose=0)
X_tsne = tsne.fit_transform(X)

print("✓ t-SNE transformation complete!")

# Create scatter plot
fig, ax = plt.subplots(figsize=(14, 10))

colors = plt.cm.tab20(np.linspace(0, 1, max(len(unique_clusters), 10)))

# Plot clusters
for idx, cluster_id in enumerate(sorted(unique_clusters)):
    if cluster_id == -1:
        # Plot outliers with X marker
        mask = clusters == cluster_id
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
                   c='red', marker='X', s=300, alpha=0.8, 
                   edgecolors='darkred', linewidth=2,
                   label=f'Outliers ({mask.sum()} points)', zorder=5)
    else:
        # Plot clusters
        mask = clusters == cluster_id
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
                   c=[colors[idx]], s=100, alpha=0.7, 
                   edgecolors='black', linewidth=0.5,
                   label=f'Cluster {int(cluster_id)} ({mask.sum()} points)', zorder=3)

ax.set_xlabel('t-SNE Dimension 1', fontsize=12, fontweight='bold')
ax.set_ylabel('t-SNE Dimension 2', fontsize=12, fontweight='bold')
ax.set_title(f'DBSCAN Clustering - t-SNE Visualization\neps={best_eps}, min_samples={best_min_samples}', 
             fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../plots/clustering/dbscan_clusters_tsne.png', dpi=150, bbox_inches='tight')
plt.close()

print("✓ t-SNE visualization saved!")

Applying t-SNE for 2D visualization (this may take 30-60 seconds)...
✓ t-SNE transformation complete!
✓ t-SNE visualization saved!


## Step 8: Export Clustered Data to CSV

In [32]:
# =========================================
# STEP 8: EXPORT CLUSTERED DATA TO CSV
# =========================================

# Create a new dataframe with cluster labels
clustered_data = df.copy()
clustered_data['cluster'] = clusters

# Identify outliers (cluster = -1)
clustered_data['is_outlier'] = (clustered_data['cluster'] == -1).astype(int)

# Reorder columns: put cluster and is_outlier after LoanApproved
col_order = ['LoanApproved', 'cluster', 'is_outlier'] + [col for col in clustered_data.columns if col not in ['LoanApproved', 'cluster', 'is_outlier']]
clustered_data = clustered_data[col_order]

# Save to CSV
output_path = '../data/processed/clustered_data.csv'
clustered_data.to_csv(output_path, index=False)

print("="*80)
print("✓ Clustered data exported successfully!")
print(f"  File: {output_path}")
print(f"  Shape: {clustered_data.shape}")
print(f"  Rows: {len(clustered_data):,} (full dataset)")
print("="*80)

print("\nFirst 10 rows of clustered data:")
print(clustered_data.head(10))

print("\nCluster value counts:")
print(clustered_data['cluster'].value_counts().sort_index())

print("\nOutlier statistics:")
print(f"  Total outliers: {clustered_data['is_outlier'].sum()}")
print(f"  Outlier percentage: {clustered_data['is_outlier'].mean()*100:.2f}%")


✓ Clustered data exported successfully!
  File: ../data/processed/clustered_data.csv
  Shape: (16770, 15)
  Rows: 16,770 (full dataset)

First 10 rows of clustered data:
   LoanApproved  cluster  is_outlier       Age    Income  LoanAmount  \
0             0        0           0  0.482105 -0.459419   -0.974577   
1             0        0           0 -0.132883 -0.467906    0.271139   
2             0        0           0  0.657816 -0.431860   -0.542205   
3             0        0           0  1.624227  0.575305    1.416370   
4             1        0           0 -0.220739  1.789160   -1.357963   
5             0        0           0  1.624227 -0.058044   -1.014288   
6             1        0           0  0.833527  1.578955   -0.348289   
7             0        0           0  0.569961 -0.435731    0.212974   
8             0       -1           1 -0.484306  0.737319    2.461600   
9             0        0           0 -0.484306 -1.345412    0.661771   

   CreditScore  YearsExperience  Empl

## Step 9: Summary Statistics

In [33]:
# =========================================
# STEP 9: SUMMARY STATISTICS
# =========================================

print("="*80)
print("CLUSTERING SUMMARY")
print("="*80)

print(f"\n1. DATA PROCESSED:")
print(f"   - Dataset size: {len(X):,} rows")
print(f"   - Features: {X.shape[1]}")
print(f"   - All data used for clustering")

print(f"\n2. SELECTED FEATURES FOR CLUSTERING:")
for i, feat in enumerate(selected_features, 1):
    print(f"   {i}. {feat}")

print(f"\n3. DBSCAN PARAMETERS:")
print(f"   - eps: {best_eps}")
print(f"   - min_samples: {best_min_samples}")

print(f"\n4. CLUSTERING RESULTS:")
print(f"   - Number of clusters: {n_clusters}")
print(f"   - Number of outliers: {n_noise}")
print(f"   - Outlier percentage: {(n_noise/len(clusters))*100:.2f}%")
if n_clusters > 0:
    cluster_sizes = [counts[i] for i, c in enumerate(unique) if c != -1]
    if len(cluster_sizes) > 0:
        print(f"   - Largest cluster: {max(cluster_sizes):,} points")
        print(f"   - Smallest cluster: {min(cluster_sizes):,} points")
        print(f"   - Average cluster size: {np.mean(cluster_sizes):.0f} points")

print(f"\n5. CLUSTERING QUALITY METRICS:")
print(f"   - Silhouette Score: {silhouette:.4f}")
print(f"     (Range: -1 to 1, where 1 = perfect, 0 = overlapping)")
print(f"   - Davies-Bouldin Index: {davies_bouldin:.4f}")
print(f"     (Lower is better, < 1.5 indicates good separation)")

print(f"\n6. CLUSTER DISTRIBUTION:")
unique, counts = np.unique(clusters, return_counts=True)
for cluster_id, count in zip(unique, counts):
    if cluster_id == -1:
        print(f"   - Outliers: {count:,} points ({(count/len(clusters))*100:.2f}%)")
    else:
        print(f"   - Cluster {int(cluster_id):2d}: {count:,} points ({(count/len(clusters))*100:.2f}%)")

print(f"\n7. VISUALIZATIONS & OUTPUT FILES:")
print(f"   ✓ data/processed/clustered_data.csv")
print(f"   ✓ plots/clustering/k_distance_graph.png")
print(f"   ✓ plots/clustering/dbscan_clusters_pca.png")
print(f"   ✓ plots/clustering/dbscan_clusters_tsne.png")

print(f"\n8. INTERPRETATION TIPS:")
print(f"   - Red X marks = outliers (potential anomalies/high-risk customers)")
print(f"   - Colored dots = behavioral groups/segments")
print(f"   - Close proximity = similar borrower profiles")
print(f"   - Isolated points = unusual financial behavior")

print("\n" + "="*80)
print("✓ CLUSTERING COMPLETE!")
print("="*80)
print("\nNEXT STEPS:")
print("  1. Review visualization plots for behavioral pattern insights")
print("  2. Analyze cluster characteristics (avg age, income, credit score, etc.)")
print("  3. Investigate outliers - may indicate fraud or default risk")
print("  4. Use clusters for customer segmentation and risk profiling")
print("  5. Compare cluster assignments with loan approval patterns")


CLUSTERING SUMMARY

1. DATA PROCESSED:
   - Dataset size: 16,770 rows
   - Features: 5
   - All data used for clustering

2. SELECTED FEATURES FOR CLUSTERING:
   1. Income
   2. LoanAmount
   3. CreditScore
   4. Age
   5. YearsExperience

3. DBSCAN PARAMETERS:
   - eps: 0.6
   - min_samples: 5

4. CLUSTERING RESULTS:
   - Number of clusters: 10
   - Number of outliers: 429
   - Outlier percentage: 2.56%
   - Largest cluster: 16,297 points
   - Smallest cluster: 3 points
   - Average cluster size: 1634 points

5. CLUSTERING QUALITY METRICS:
   - Silhouette Score: -0.0349
     (Range: -1 to 1, where 1 = perfect, 0 = overlapping)
   - Davies-Bouldin Index: 1.5981
     (Lower is better, < 1.5 indicates good separation)

6. CLUSTER DISTRIBUTION:
   - Outliers: 429 points (2.56%)
   - Cluster  0: 16,297 points (97.18%)
   - Cluster  1: 8 points (0.05%)
   - Cluster  2: 4 points (0.02%)
   - Cluster  3: 5 points (0.03%)
   - Cluster  4: 6 points (0.04%)
   - Cluster  5: 5 points (0.03%)
   -